> **Attribution:** Adapted from [Keithland89/StudioLens-for-Copilot-Studio](https://github.com/Keithland89/StudioLens-for-Copilot-Studio), licensed under the MIT License.
>
> Adapted by the ESS Insights team for ESS-only scoping.

# Copilot Credit Consumption — Ingester (Fabric)

Lands the three **Power Platform admin center Copilot Credits reports** (`EntitlementConsumption*_MCSMessages_*.csv`) into Lakehouse Delta tables for observed aggregate credit reporting.

The notebook reads files already landed in the Lakehouse; it does not acquire [Power Platform admin center](https://admin.powerplatform.microsoft.com) exports. Upload all three files manually to `Files/credit_consumption`, or implement a separate landing pipeline. No companion Power Automate flow is included in this repository.

```
Power Platform admin center  ─(download)─▶  manual/automated landing  ─▶  Lakehouse Files/credit_consumption/*.csv  ─▶  THIS notebook  ─▶  Delta
```

| Source CSV (prefix) | → Delta table | Grain |
| --- | --- | --- |
| `EntitlementConsumptionTenantDetailsReport_MCSMessages` | `credit_consumption_tenant` | environment × capacity × usage date (daily) |
| `EntitlementConsumptionTenantPerAgentDetailsReport_MCSMessages` | `credit_consumption_agent` | agent × feature × environment × lookback snapshot |
| `EntitlementConsumptionTenantPerUserDetailsReport_MCSMessages` | `credit_consumption_user` | user × agent × lookback snapshot |
| unmapped agent/user rows | `credit_consumption_quarantine` | identifier-only registry diagnostics |

**Fail-safe ingestion** — all three reports are required. Missing, duplicate, header-only, or malformed inputs fail during preflight before any Delta table is overwritten, so a failed run cannot erase or mix prior valid snapshots.

**Multi-environment** — each tenant-level Power Platform admin center export already carries all environments and the Agent / Tenant reports include `Environment Id` / `Environment Name`. Keep exactly one current file per report type in `Files/credit_consumption/`; multiple overlapping snapshots are rejected to prevent double counting.

**No app registration / Graph permission required** — this notebook only reads files already in the Lakehouse.

**Attribution boundary** — the Power Platform admin center exports have no conversation/session ID. Observed credits are scoped through `CreditId` registry aliases and joined through `canonical_agent_key`; user-report rows never infer an environment from agent name; any per-conversation, resolved-conversation, topic, or outcome value is a modeled allocation and must be labeled as such. Blank `User Email` rows remain in the user table as unmatched identity.

## 1. Configuration

Toggle this as the pipeline parameter cell when orchestration overrides `WRITE_MODE`, `AGENT_SCOPE_MODE`, or `OUTPUT_PREFIX`. Leave `OUTPUT_PREFIX=''` for a standard Lakehouse; use `dbo` only when schemas are enabled. The same validated agent registry used by the transcript parser is required.


In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

SOURCE_DIR = 'Files/credit_consumption'
REPORTS = {
    'credit_consumption_tenant': 'EntitlementConsumptionTenantDetailsReport_MCSMessages*',
    'credit_consumption_agent':  'EntitlementConsumptionTenantPerAgentDetailsReport_MCSMessages*',
    'credit_consumption_user':   'EntitlementConsumptionTenantPerUserDetailsReport_MCSMessages*',
}
AGENT_REGISTRY_FILE = '/lakehouse/default/Files/config/agent-registry.csv'
AGENT_SCOPE_MODE = 'ESS Safe'
OUTPUT_PREFIX = ''
WRITE_MODE = 'overwrite'
ADD_LINEAGE = True
STRICT = True

if AGENT_SCOPE_MODE not in {'ESS Safe', 'Selected Agents', 'All Agents'}:
    raise ValueError(
        'AGENT_SCOPE_MODE must be exactly one of: ESS Safe, Selected Agents, All Agents')


## 2. Helpers

`_list_matches` resolves a glob against the Lakehouse `Files/` area (via `notebookutils.fs.ls` (mount-independent)). `_read_csv` parses with multiline + quote-escape so the reports' free-text columns (knowledge sources, scenario names) survive. `_sanitize` makes column names Delta-safe while the PBIP keeps reading the original names through Delta column-mapping.

In [ ]:
import os, glob, re, datetime, csv, hashlib
from pyspark.sql import functions as F

_LOAD_TS = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
_SCOPE_MODES = {'ESS Safe', 'Selected Agents', 'All Agents'}
_ALIAS_TYPES = {'TranscriptId', 'TranscriptSchema', 'CreditId', 'M365Title'}
_REGISTRY_COLUMNS = {
    'RegistryVersion', 'AgentKey', 'AgentLabel', 'IsEss', 'Selected', 'Enabled',
    'EnvironmentKey', 'EnvironmentName', 'EnvironmentUrl', 'AliasType', 'AliasValue'}

def _local(path):
    return path if path.startswith('/') else f'/lakehouse/default/{path}'

def _table(name):
    return f'{OUTPUT_PREFIX}.{name}' if OUTPUT_PREFIX else name

def _list_matches(pattern):
    rx = re.compile('^' + re.escape(pattern).replace('\\*', '.*') + '$', re.IGNORECASE)
    try:
        entries = notebookutils.fs.ls(SOURCE_DIR)
    except Exception:
        return []
    hits = [
        entry.name for entry in entries
        if not entry.isDir and rx.match(entry.name) and entry.name.lower().endswith('.csv')]
    return [f'{SOURCE_DIR}/{name}' for name in sorted(hits)]

def _read_csv(path):
    return (spark.read.option('header', True).option('multiLine', True)
            .option('escape', '"').option('encoding', 'UTF-8').csv(path))

_INVALID = re.compile(r'[ ,;{}()\n\t=/-]')
def _sanitize(df):
    return df.toDF(*[_INVALID.sub('_', column.lstrip('\ufeff')) for column in df.columns])

def normalize_agent_alias(alias_type, value):
    normalized = str(value or '').strip()
    if alias_type in {'TranscriptId', 'CreditId'}:
        normalized = normalized.strip('{}')
        if alias_type == 'CreditId' and normalized.casefold().startswith('p_'):
            normalized = normalized[2:]
    return normalized.casefold()

def _registry_bool(value, field):
    normalized = str(value or '').strip().casefold()
    if normalized in {'true', '1', 'yes'}:
        return True
    if normalized in {'false', '0', 'no'}:
        return False
    raise ValueError(f'{field} must be true or false, not {value!r}')

def load_agent_registry(path):
    try:
        with open(path, 'r', encoding='utf-8-sig', newline='') as handle:
            reader = csv.DictReader(handle)
            if not reader.fieldnames:
                raise ValueError(f'Agent registry has no header: {path}')
            missing = sorted(_REGISTRY_COLUMNS - set(reader.fieldnames))
            if missing:
                raise ValueError(f'Agent registry is missing columns: {missing}')
            source_rows = list(reader)
    except FileNotFoundError as error:
        raise ValueError(f'Agent registry not found: {path}') from error
    rows, metadata_by_key, aliases = [], {}, {}
    for row_number, row in enumerate(source_rows, start=2):
        version = str(row.get('RegistryVersion') or '').strip()
        agent_key = str(row.get('AgentKey') or '').strip()
        agent_label = str(row.get('AgentLabel') or '').strip()
        alias_type = str(row.get('AliasType') or '').strip()
        alias_value = str(row.get('AliasValue') or '').strip()
        environment_key = str(row.get('EnvironmentKey') or '').strip()
        if version != '1':
            raise ValueError(
                f'Agent registry row {row_number} has unsupported version {version!r}')
        if not agent_key or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._:-]*', agent_key):
            raise ValueError(f'Agent registry row {row_number} has invalid AgentKey')
        if not agent_label or alias_type not in _ALIAS_TYPES:
            raise ValueError(f'Agent registry row {row_number} is invalid')
        alias_normalized = normalize_agent_alias(alias_type, alias_value)
        if not alias_normalized:
            raise ValueError(f'Agent registry row {row_number} has no AliasValue')
        normalized = {
            'RegistryVersion': '1', 'AgentKey': agent_key,
            'AgentLabel': agent_label,
            'IsEss': _registry_bool(row.get('IsEss'), 'IsEss'),
            'Selected': _registry_bool(row.get('Selected'), 'Selected'),
            'Enabled': _registry_bool(row.get('Enabled'), 'Enabled'),
            'EnvironmentKey': environment_key,
            'EnvironmentName': str(row.get('EnvironmentName') or '').strip(),
            'EnvironmentUrl': str(row.get('EnvironmentUrl') or '').strip(),
            'AliasType': alias_type, 'AliasValue': alias_value,
            'AliasValueNormalized': alias_normalized}
        metadata = (
            normalized['AgentLabel'], normalized['IsEss'],
            normalized['Selected'], normalized['Enabled'])
        if metadata_by_key.setdefault(agent_key.casefold(), metadata) != metadata:
            raise ValueError(
                f'Agent registry has conflicting metadata for AgentKey {agent_key!r}')
        alias_key = (
            environment_key.casefold(), alias_type.casefold(), alias_normalized)
        if aliases.setdefault(alias_key, agent_key).casefold() != agent_key.casefold():
            raise ValueError(
                'Agent registry alias maps to multiple agents: '
                f'{environment_key or "*"} / {alias_type} / {alias_value}')
        rows.append(normalized)
    if not rows:
        raise ValueError('Agent registry has no rows')
    return rows

def active_agent_keys(rows, scope_mode):
    if scope_mode not in _SCOPE_MODES:
        raise ValueError(
            'AGENT_SCOPE_MODE must be exactly one of: ESS Safe, Selected Agents, All Agents')
    keys = {
        row['AgentKey'] for row in rows
        if row['Enabled']
        and (scope_mode != 'ESS Safe' or row['IsEss'])
        and (scope_mode != 'Selected Agents' or row['Selected'])}
    if scope_mode != 'All Agents' and not keys:
        raise ValueError(
            f'Agent registry has no enabled agents for scope mode {scope_mode!r}')
    return keys

def registry_environment_key(rows, environment_id='', environment_name=''):
    raw_values = {
        str(value or '').strip().rstrip('/').casefold()
        for value in (environment_id, environment_name) if str(value or '').strip()}
    if not raw_values:
        return ''
    keys = {
        row['EnvironmentKey'] for row in rows
        if row['Enabled'] and any(
            str(row[field] or '').strip().rstrip('/').casefold() in raw_values
            for field in ('EnvironmentKey', 'EnvironmentName', 'EnvironmentUrl'))}
    if len(keys) > 1:
        raise ValueError(
            f'Agent registry environment is ambiguous: {environment_id!r} / {environment_name!r}')
    return next(iter(keys), '')

def resolve_registry_alias(
        rows, alias_type, alias_value, active_keys=None, environment_key=''):
    normalized = normalize_agent_alias(alias_type, alias_value)
    if not normalized:
        return None
    candidates = [
        row for row in rows
        if row['Enabled'] and row['AliasType'] == alias_type
        and row['AliasValueNormalized'] == normalized
        and (active_keys is None or row['AgentKey'] in active_keys)]
    environment = str(environment_key or '').strip().casefold()
    if environment:
        exact = [
            row for row in candidates
            if row['EnvironmentKey'].casefold() == environment]
        candidates = exact or [
            row for row in candidates if not row['EnvironmentKey']]
    keys = {row['AgentKey'] for row in candidates}
    if len(keys) > 1:
        raise ValueError(
            f'Agent alias is ambiguous without an environment: {alias_type} / {alias_value}')
    return candidates[0] if candidates else None

def _raw_key(kind, value, environment=''):
    material = (
        f'{str(environment or "").strip().casefold()}|{kind.casefold()}|'
        f'{normalize_agent_alias(kind, value)}')
    return (
        f'raw:{kind.casefold()}:'
        f'{hashlib.sha256(material.encode("utf-8")).hexdigest()[:24]}')

def resolve_credit_agent(
        agent_id, environment_id='', environment_name='',
        registry_rows=None, active_keys=None, scope_mode=None):
    registry_rows = AGENT_REGISTRY_ROWS if registry_rows is None else registry_rows
    active_keys = ACTIVE_AGENT_KEYS if active_keys is None else active_keys
    scope_mode = AGENT_SCOPE_MODE if scope_mode is None else scope_mode
    raw_id = str(agent_id or '').strip()
    raw_environment_id = str(environment_id or '').strip()
    raw_environment_name = str(environment_name or '').strip()
    environment_key = registry_environment_key(
        registry_rows, raw_environment_id, raw_environment_name)
    match = resolve_registry_alias(
        registry_rows, 'CreditId', raw_id, active_keys, environment_key)
    if match:
        return {
            'canonical_agent_key': match['AgentKey'],
            'canonical_agent_name': match['AgentLabel'],
            'canonical_environment_key': (
                match['EnvironmentKey'] if environment_key else ''),
            'agent_mapping_status': 'RegistryMapped',
            'raw_agent_id': raw_id,
            'raw_environment_id': raw_environment_id,
            'raw_environment_name': raw_environment_name}
    if scope_mode == 'All Agents' and raw_id:
        return {
            'canonical_agent_key': _raw_key(
                'CreditId', raw_id, raw_environment_id),
            'canonical_agent_name': raw_id,
            'canonical_environment_key': (
                environment_key
                or (_raw_key('Environment', raw_environment_id)
                    if raw_environment_id else '')),
            'agent_mapping_status': 'RawAllAgents',
            'raw_agent_id': raw_id,
            'raw_environment_id': raw_environment_id,
            'raw_environment_name': raw_environment_name}
    return {
        'canonical_agent_key': '',
        'canonical_agent_name': '',
        'canonical_environment_key': '',
        'agent_mapping_status': 'UnmappedQuarantined',
        'raw_agent_id': raw_id,
        'raw_environment_id': raw_environment_id,
        'raw_environment_name': raw_environment_name}

AGENT_REGISTRY_ROWS = load_agent_registry(AGENT_REGISTRY_FILE)
ACTIVE_AGENT_KEYS = active_agent_keys(AGENT_REGISTRY_ROWS, AGENT_SCOPE_MODE)
print(f'Load timestamp: {_LOAD_TS}')
print(f'Source folder : {SOURCE_DIR}  (exists: {os.path.isdir(_local(SOURCE_DIR))})')
print(
    f'Agent registry: {len(AGENT_REGISTRY_ROWS):,} aliases; '
    f'{len(ACTIVE_AGENT_KEYS):,} active agent(s); scope={AGENT_SCOPE_MODE}')


## 3. Ingest each report → Delta

Loops the `REPORTS` map. For each target table it requires exactly one nonempty current CSV, validates required headers, explicitly casts quantitative fields to `decimal(38,6)`, adds lineage columns, sanitises, and materialises the row count. Only after all three reports pass does it write any Delta table.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DecimalType)

EMPTY_SCHEMAS = {
    'credit_consumption_tenant': [
        'BillingPlan_Id','BillingPlan_Name','Environment_Id','Environment_Name',
        'Capacity_Type','Entitled_Quantity','Prepaid_Consumed_Quantity',
        'Pay_as_you_go_Consumed_Quantity','Usage_Date'],
    'credit_consumption_agent': [
        'Agent_Name','Agent_Id','Product','AI_Feature_Billable_Feature','Billed_credit',
        'Non_billed_credit','Channel','Knowledge_Sources','Tool_Used','LLM_Model',
        'Scenario_Name','Environment_Id','Environment_Name'],
    'credit_consumption_user': [
        'User_Id','User_Email','Agent_Id','Agent_Name','Billable_credit_used',
        'Credits_used','M365_Copilot_Licensed'],
}
CANONICAL_CREDIT_COLUMNS = [
    'canonical_agent_key', 'canonical_agent_name', 'canonical_environment_key',
    'agent_mapping_status', 'raw_agent_id', 'raw_environment_id',
    'raw_environment_name']
NUMERIC_COLUMNS = {
    'credit_consumption_tenant': [
        'Entitled_Quantity','Prepaid_Consumed_Quantity',
        'Pay_as_you_go_Consumed_Quantity'],
    'credit_consumption_agent': ['Billed_credit','Non_billed_credit'],
    'credit_consumption_user': ['Billable_credit_used','Credits_used'],
}

def _empty(columns, canonical=False):
    columns = list(columns) + (CANONICAL_CREDIT_COLUMNS if canonical else [])
    if ADD_LINEAGE:
        columns += ['SourceFile', 'LoadDate']
    numeric = {column for values in NUMERIC_COLUMNS.values() for column in values}
    return spark.createDataFrame([], StructType([
        StructField(
            column,
            DecimalType(38, 6) if column in numeric else StringType(),
            True)
        for column in columns]))

def _validate_and_cast(df, logical_table, path):
    missing = sorted(set(EMPTY_SCHEMAS[logical_table]) - set(df.columns))
    if missing:
        raise ValueError(
            f'{logical_table}: matched file "{path}" is malformed; '
            'missing required column(s) after sanitization: ' + ', '.join(missing))
    for column in NUMERIC_COLUMNS[logical_table]:
        df = df.withColumn(column, F.col(column).cast(DecimalType(38, 6)))
    return df

_identity_schema = StructType([
    StructField(column, StringType(), True)
    for column in CANONICAL_CREDIT_COLUMNS])

def _map_credit_rows(df, logical_table):
    if logical_table == 'credit_consumption_tenant':
        return df, None
    has_environment = logical_table == 'credit_consumption_agent'
    resolver = F.udf(
        lambda agent_id, environment_id, environment_name: resolve_credit_agent(
            agent_id, environment_id, environment_name),
        _identity_schema)
    environment_id = (
        F.col('Environment_Id') if has_environment else F.lit(''))
    environment_name = (
        F.col('Environment_Name') if has_environment else F.lit(''))
    mapped = df.withColumn(
        '_agent_identity',
        resolver(F.col('Agent_Id'), environment_id, environment_name))
    for column in CANONICAL_CREDIT_COLUMNS:
        mapped = mapped.withColumn(
            column, F.col(f'_agent_identity.{column}'))
    mapped = mapped.drop('_agent_identity')
    quarantine = (
        mapped.filter(F.col('agent_mapping_status') == 'UnmappedQuarantined')
        .select(
            F.lit(logical_table).alias('source_report'),
            'raw_agent_id', 'raw_environment_id', 'raw_environment_name',
            'agent_mapping_status'))
    curated = mapped.filter(
        F.col('agent_mapping_status') != 'UnmappedQuarantined')
    if (AGENT_SCOPE_MODE != 'All Agents'
            and df.limit(1).count() > 0
            and curated.limit(1).count() == 0):
        raise ValueError(
            f'{logical_table}: no credit agents matched '
            f'AGENT_SCOPE_MODE={AGENT_SCOPE_MODE!r}; no Delta table was changed.')
    return curated, quarantine

def _write(df, logical_table):
    table = _table(logical_table)
    (df.write.mode(WRITE_MODE)
        .option('overwriteSchema', 'true')
        .option('delta.columnMapping.mode', 'name')
        .option('delta.minReaderVersion', '2')
        .option('delta.minWriterVersion', '5')
        .format('delta').saveAsTable(table))

if WRITE_MODE == 'append':
    print(
        '⚠  APPEND stores snapshot history: overlapping rolling windows are not '
        'transactions. Use LoadDate-aware selection/deduplication and never sum '
        'snapshots blindly.')

prepared, quarantines = {}, []
for logical_table, pattern in REPORTS.items():
    matches = _list_matches(pattern)
    if not matches:
        raise FileNotFoundError(
            f'{logical_table}: no file matched "{pattern}". All three current '
            'Power Platform admin center exports are required; no existing Delta '
            'table was changed.')
    if len(matches) > 1:
        raise ValueError(
            f'{logical_table}: {len(matches)} files matched "{pattern}". Keep '
            'exactly one current tenant-level snapshot per report type; overlapping '
            'lookback exports cannot be unioned without double counting. Matched '
            'files: ' + ', '.join(matches))
    path = matches[0]
    df = _validate_and_cast(
        _sanitize(_read_csv(path)), logical_table, path)
    if ADD_LINEAGE:
        df = (
            df.withColumn('SourceFile', F.lit(os.path.basename(path)))
            .withColumn('LoadDate', F.lit(_LOAD_TS)))
    curated, quarantine = _map_credit_rows(df, logical_table)
    if quarantine is not None:
        quarantines.append(quarantine)
    prepared[logical_table] = (curated, 1, curated.count())

print('✓  Preflight passed for every report; starting Delta writes.')
summary = []
for logical_table in REPORTS:
    df, files, count = prepared[logical_table]
    _write(df, logical_table)
    summary.append((logical_table, files, count))
    print(
        f'✓  {_table(logical_table):34} — 1 current snapshot, '
        f'{count:,} scoped rows -> written ({WRITE_MODE})')

if quarantines:
    quarantine = quarantines[0]
    for frame in quarantines[1:]:
        quarantine = quarantine.unionByName(frame)
    quarantine = quarantine.dropDuplicates([
        'source_report', 'raw_agent_id',
        'raw_environment_id', 'raw_environment_name'])
else:
    quarantine = spark.createDataFrame([], StructType([
        StructField('source_report', StringType(), True),
        StructField('raw_agent_id', StringType(), True),
        StructField('raw_environment_id', StringType(), True),
        StructField('raw_environment_name', StringType(), True),
        StructField('agent_mapping_status', StringType(), True)]))
_write(quarantine, 'credit_consumption_quarantine')
print(
    f'✓  {_table("credit_consumption_quarantine"):34} — '
    f'{quarantine.count():,} identifier-only row(s)')
print(
    '\nDone. Credit-consumption Delta tables written. Refresh the supplied '
    'Import model through its SQL analytics endpoint to pick them up.')


## 4. Verify + quick reconciliation

Prints observed Power Platform admin center headline totals and, if `agent_sessions` exists, the transcript-native diagnostic total side by side. This is not an exact reconciliation: the export has no conversation/session ID, and per-conversation or resolved-conversation attribution is modeled only.

In [ ]:
def _num(col):
    return F.sum(F.coalesce(F.col(col).cast('double'), F.lit(0.0)))

for table, files, rows in summary:
    print(f'{table:34} files={files} rows={rows:,}')

try:
    agent = spark.table(_table('credit_consumption_agent'))
    if agent.count():
        tot = agent.select(_num('Billed_credit').alias('billed'),
                           _num('Non_billed_credit').alias('nonbilled')).collect()[0]
        print(f"\nObserved Power Platform admin center credits — billed: {tot['billed']:,.2f}   non-billed: {tot['nonbilled']:,.2f}")
except Exception as e:
    print('agent consumption summary skipped:', e)

try:
    user = spark.table(_table('credit_consumption_user'))
    unmatched = user.filter(F.col('User_Email').isNull() | (F.trim(F.col('User_Email')) == '')).count()
    print(f'User rows with unmatched identity (blank User Email), retained: {unmatched:,}')
except Exception as e:
    print('user identity summary skipped:', e)

try:
    sess = spark.table(_table('agent_sessions'))
    disp = sess.select(_num('total_displayed_cost').alias('disp')).collect()[0]['disp']
    print(f"Transcript diagnostic — displayedCost credits: {disp:,.2f}")
    print('(Diagnostic comparison only; the Power Platform admin center export supplies no conversation/session ID. Do not treat this as exact billing attribution.)')
except Exception:
    print('(agent_sessions not present yet — run the transcript parser to enable reconciliation.)')

---
**Connect the PBIP**: these three tables feed the semantic-model queries `Credit Consumption (Agent)`, `Credit Consumption (User)`, `Credit Consumption (Tenant)`, all gated by the `Enable_Consumption` parameter. Leave `Enable_Consumption = false` for customers who don't supply the reports and the billing visuals stay dormant — the transcript-native `Total Cost Units` measure keeps working regardless.